In [1]:
import numpy as np
import pandas as pd
from datasets import load_dataset

REPO_ID = "witgaw/METR-LA"

train = load_dataset(
    REPO_ID,
    split="train"
)

print(train)

assert train.num_rows > 0
assert "node_id" in train.column_names
assert "t0_timestamp" in train.column_names

print("\nDataset loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Dataset({
    features: ['node_id', 't0_timestamp', 'x_t-11_d0', 'x_t-11_d1', 'x_t-10_d0', 'x_t-10_d1', 'x_t-9_d0', 'x_t-9_d1', 'x_t-8_d0', 'x_t-8_d1', 'x_t-7_d0', 'x_t-7_d1', 'x_t-6_d0', 'x_t-6_d1', 'x_t-5_d0', 'x_t-5_d1', 'x_t-4_d0', 'x_t-4_d1', 'x_t-3_d0', 'x_t-3_d1', 'x_t-2_d0', 'x_t-2_d1', 'x_t-1_d0', 'x_t-1_d1', 'x_t+0_d0', 'x_t+0_d1', 'y_t+1_d0', 'y_t+1_d1', 'y_t+2_d0', 'y_t+2_d1', 'y_t+3_d0', 'y_t+3_d1', 'y_t+4_d0', 'y_t+4_d1', 'y_t+5_d0', 'y_t+5_d1', 'y_t+6_d0', 'y_t+6_d1', 'y_t+7_d0', 'y_t+7_d1', 'y_t+8_d0', 'y_t+8_d1', 'y_t+9_d0', 'y_t+9_d1', 'y_t+10_d0', 'y_t+10_d1', 'y_t+11_d0', 'y_t+11_d1', 'y_t+12_d0', 'y_t+12_d1'],
    num_rows: 4962618
})

Dataset loaded successfully.


In [2]:
columns = train.column_names

speed_cols = [
    col for col in columns
    if (
        (col.startswith("x_t") or col.startswith("y_t"))
        and col.endswith("_d0")
    )
]

print("Speed columns:", len(speed_cols))

for col in speed_cols:
    print(col)

assert len(speed_cols) > 0, "No speed columns detected."
assert "node_id" not in speed_cols
assert "t0_timestamp" not in speed_cols
assert all(col.endswith("_d0") for col in speed_cols)

print("\nPASS: Only speed columns selected.")
print("node_id and other identifiers explicitly excluded.")

Speed columns: 24
x_t-11_d0
x_t-10_d0
x_t-9_d0
x_t-8_d0
x_t-7_d0
x_t-6_d0
x_t-5_d0
x_t-4_d0
x_t-3_d0
x_t-2_d0
x_t-1_d0
x_t+0_d0
y_t+1_d0
y_t+2_d0
y_t+3_d0
y_t+4_d0
y_t+5_d0
y_t+6_d0
y_t+7_d0
y_t+8_d0
y_t+9_d0
y_t+10_d0
y_t+11_d0
y_t+12_d0

PASS: Only speed columns selected.
node_id and other identifiers explicitly excluded.


In [3]:
speed_df = train.select_columns(
    speed_cols
).to_pandas()

print("Speed-only shape:", speed_df.shape)

speed_summary = speed_df.describe().T

speed_summary = speed_summary[
    [
        "mean",
        "std",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    ]
]

display(speed_summary)

assert "node_id" not in speed_summary.index
assert "t0_timestamp" not in speed_summary.index

assert list(speed_summary.columns) == [
    "mean",
    "std",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
]

print("\nPASS: Speed summary statistics computed.")

Speed-only shape: (4962618, 24)


,mean,std,min,25%,50%,75%,max
x_t-11_d0,54.402867,19.497466,0.0,54.0,62.555556,66.25,70.0
x_t-10_d0,54.402913,19.497441,0.0,54.0,62.555556,66.25,70.0
x_t-9_d0,54.402931,19.497405,0.0,54.0,62.555556,66.25,70.0
x_t-8_d0,54.402944,19.497388,0.0,54.0,62.555556,66.25,70.0
x_t-7_d0,54.405487,19.494383,0.0,54.0,62.555556,66.25,70.0
x_t-6_d0,54.407999,19.491387,0.0,54.0,62.555556,66.25,70.0
x_t-5_d0,54.407881,19.491388,0.0,54.0,62.555556,66.25,70.0
x_t-4_d0,54.407807,19.491422,0.0,54.0,62.555556,66.25,70.0
x_t-3_d0,54.407717,19.491504,0.0,54.0,62.555556,66.25,70.0
x_t-2_d0,54.407625,19.491599,0.0,54.0,62.555556,66.25,70.0



PASS: Speed summary statistics computed.


In [4]:
ZERO_GAP_THRESHOLD = 20.0

speed_summary["zero_to_q1_gap"] = (
    speed_summary["25%"]
    - speed_summary["min"]
)

speed_summary["stuck_zero_pattern"] = (
    np.isclose(speed_summary["min"], 0.0)
    &
    (
        speed_summary["zero_to_q1_gap"]
        > ZERO_GAP_THRESHOLD
    )
)

flagged_speed_cols = speed_summary[
    speed_summary["stuck_zero_pattern"]
]

print("Columns showing suspicious zero-minimum pattern:")
print("Count:", len(flagged_speed_cols))

display(
    flagged_speed_cols[
        [
            "min",
            "25%",
            "zero_to_q1_gap"
        ]
    ]
)

Columns showing suspicious zero-minimum pattern:
Count: 24


,min,25%,zero_to_q1_gap
x_t-11_d0,0.0,54.0,54.0
x_t-10_d0,0.0,54.0,54.0
x_t-9_d0,0.0,54.0,54.0
x_t-8_d0,0.0,54.0,54.0
x_t-7_d0,0.0,54.0,54.0
x_t-6_d0,0.0,54.0,54.0
x_t-5_d0,0.0,54.0,54.0
x_t-4_d0,0.0,54.0,54.0
x_t-3_d0,0.0,54.0,54.0
x_t-2_d0,0.0,54.0,54.0


In [5]:
if len(flagged_speed_cols) > 0:

    print("=" * 65)
    print("WARNING: STUCK-AT-ZERO PATTERN DETECTED")
    print("=" * 65)

    print(
        f"{len(flagged_speed_cols)} speed column(s) have a "
        "minimum of 0.0 while their 25th percentile is "
        f"more than {ZERO_GAP_THRESHOLD:.0f} mph above zero."
    )

    print(
        "\nThis is consistent with the known METR-LA "
        "stuck-at-zero / failed-sensor reading pattern "
        "rather than ordinary low-speed traffic alone."
    )

    print("\nAffected columns:")

    for col, row in flagged_speed_cols.iterrows():

        print(
            f"  {col:15s} "
            f"min={row['min']:.2f}, "
            f"Q1={row['25%']:.2f}, "
            f"gap={row['zero_to_q1_gap']:.2f}"
        )

else:

    print(
        "No speed columns satisfy the specified "
        "stuck-at-zero warning criterion."
    )

24 speed column(s) have a minimum of 0.0 while their 25th percentile is more than 20 mph above zero.

This is consistent with the known METR-LA stuck-at-zero / failed-sensor reading pattern rather than ordinary low-speed traffic alone.

Affected columns:
  x_t-11_d0       min=0.00, Q1=54.00, gap=54.00
  x_t-10_d0       min=0.00, Q1=54.00, gap=54.00
  x_t-9_d0        min=0.00, Q1=54.00, gap=54.00
  x_t-8_d0        min=0.00, Q1=54.00, gap=54.00
  x_t-7_d0        min=0.00, Q1=54.00, gap=54.00
  x_t-6_d0        min=0.00, Q1=54.00, gap=54.00
  x_t-5_d0        min=0.00, Q1=54.00, gap=54.00
  x_t-4_d0        min=0.00, Q1=54.00, gap=54.00
  x_t-3_d0        min=0.00, Q1=54.00, gap=54.00
  x_t-2_d0        min=0.00, Q1=54.00, gap=54.00
  x_t-1_d0        min=0.00, Q1=54.00, gap=54.00
  x_t+0_d0        min=0.00, Q1=54.00, gap=54.00
  y_t+1_d0        min=0.00, Q1=54.00, gap=54.00
  y_t+2_d0        min=0.00, Q1=54.00, gap=54.00
  y_t+3_d0        min=0.00, Q1=54.00, gap=54.00
  y_t+4_d0        min=0.0

In [6]:
if "x_t+0_d0" in columns:
    CURRENT_SPEED = "x_t+0_d0"

elif "x_t-0_d0" in columns:
    CURRENT_SPEED = "x_t-0_d0"

else:
    raise AssertionError(
        "Current speed column not found."
    )


if "y_t+1_d0" in columns:
    NEXT_SPEED = "y_t+1_d0"

else:
    raise AssertionError(
        "y_t+1_d0 target column not found."
    )


print("Persistence prediction :", CURRENT_SPEED)
print("Actual next speed      :", NEXT_SPEED)

assert CURRENT_SPEED in columns
assert NEXT_SPEED in columns
assert CURRENT_SPEED != NEXT_SPEED

Persistence prediction : x_t+0_d0
Actual next speed      : y_t+1_d0


In [7]:
baseline_df = train.select_columns(
    [
        "node_id",
        CURRENT_SPEED,
        NEXT_SPEED
    ]
).to_pandas()

baseline_df = baseline_df.rename(
    columns={
        CURRENT_SPEED: "current_speed",
        NEXT_SPEED: "next_speed"
    }
)

print("Shape:", baseline_df.shape)
display(baseline_df.head())

assert baseline_df["node_id"].nunique() == 207

Shape: (4962618, 3)


,node_id,current_speed,next_speed
0,0,62.250,61.125
1,1,67.750,67.000
2,2,66.875,58.500
3,3,60.000,62.250
4,4,64.750,66.375


In [8]:
baseline_df["absolute_error"] = np.abs(
    baseline_df["next_speed"]
    - baseline_df["current_speed"]
)

assert (
    baseline_df["absolute_error"] >= 0
).all()

print(
    "Overall persistence MAE:",
    baseline_df["absolute_error"].mean()
)

Overall persistence MAE: 2.9173841491863097


In [9]:
sensor_mae = (
    baseline_df
    .groupby("node_id", as_index=False)
    .agg(
        persistence_mae=(
            "absolute_error",
            "mean"
        ),
        observations=(
            "absolute_error",
            "size"
        )
    )
)

print("Sensors:", len(sensor_mae))

display(sensor_mae.head())

assert len(sensor_mae) == 207
assert sensor_mae["node_id"].nunique() == 207
assert (sensor_mae["persistence_mae"] >= 0).all()

print("\nPASS: Per-sensor persistence MAE computed.")

Sensors: 207


,node_id,persistence_mae,observations
0,0,2.621120,23974
1,1,2.150621,23974
2,2,2.157316,23974
3,3,3.006879,23974
4,4,3.832567,23974



PASS: Per-sensor persistence MAE computed.


In [10]:
mae_distribution = (
    sensor_mae["persistence_mae"]
    .describe()
)

print("PER-SENSOR PERSISTENCE MAE DISTRIBUTION")
print("=" * 55)

print(mae_distribution)

PER-SENSOR PERSISTENCE MAE DISTRIBUTION
count    207.000000
mean       2.917384
std        0.638720
min        1.808754
25%        2.421522
50%        2.831969
75%        3.343815
max        5.135753
Name: persistence_mae, dtype: float64


In [11]:
mae_mean = sensor_mae[
    "persistence_mae"
].mean()

mae_std = sensor_mae[
    "persistence_mae"
].std()

assert mae_mean > 0, (
    "Mean persistence MAE must be > 0 "
    "to calculate coefficient of variation."
)

mae_cv = mae_std / mae_mean

print("Mean sensor MAE :", round(mae_mean, 6))
print("Std sensor MAE  :", round(mae_std, 6))
print("CV = std / mean :", round(mae_cv, 6))
print("CV (%)          :", f"{mae_cv * 100:.2f}%")

assert np.isfinite(mae_cv)
assert mae_cv >= 0

Mean sensor MAE : 2.917384
Std sensor MAE  : 0.63872
CV = std / mean : 0.218936
CV (%)          : 21.89%


In [12]:
print("=" * 60)
print("FORECAST-DIFFICULTY VARIATION")
print("=" * 60)

print(f"Coefficient of variation: {mae_cv:.4f}")
print(f"CV percentage           : {mae_cv * 100:.2f}%")

if mae_cv < 0.10:

    print(
        "\nPersistence error is relatively uniform "
        "across the sensor network."
    )

elif mae_cv < 0.25:

    print(
        "\nThere is moderate sensor-to-sensor variation "
        "in persistence forecast difficulty."
    )

else:

    print(
        "\nThere is substantial sensor-to-sensor variation "
        "in persistence forecast difficulty."
    )

print(
    "\nNote: These thresholds are descriptive heuristics, "
    "not formal statistical significance thresholds."
)

FORECAST-DIFFICULTY VARIATION
Coefficient of variation: 0.2189
CV percentage           : 21.89%

There is moderate sensor-to-sensor variation in persistence forecast difficulty.

Note: These thresholds are descriptive heuristics, not formal statistical significance thresholds.


In [13]:
lowest_5 = (
    sensor_mae
    .nsmallest(
        5,
        "persistence_mae"
    )
    .reset_index(drop=True)
)

print("5 SENSORS WITH LOWEST PERSISTENCE ERROR")
print("=" * 55)

display(lowest_5)

assert len(lowest_5) == 5

5 SENSORS WITH LOWEST PERSISTENCE ERROR


,node_id,persistence_mae,observations
0,16,1.808754,23974
1,74,1.832561,23974
2,107,1.898510,23974
3,123,1.906989,23974
4,183,1.974499,23974


In [14]:
highest_5 = (
    sensor_mae
    .nlargest(
        5,
        "persistence_mae"
    )
    .reset_index(drop=True)
)

print("5 SENSORS WITH HIGHEST PERSISTENCE ERROR")
print("=" * 55)

display(highest_5)

assert len(highest_5) == 5

5 SENSORS WITH HIGHEST PERSISTENCE ERROR


,node_id,persistence_mae,observations
0,140,5.135753,23974
1,62,4.800096,23974
2,30,4.796995,23974
3,126,4.732942,23974
4,51,4.681790,23974


In [15]:
lowest_followup = lowest_5.copy()
lowest_followup["group"] = "Lowest Error"

highest_followup = highest_5.copy()
highest_followup["group"] = "Highest Error"

followup_sensors = pd.concat(
    [
        highest_followup,
        lowest_followup
    ],
    ignore_index=True
)

followup_sensors = followup_sensors[
    [
        "node_id",
        "group",
        "persistence_mae",
        "observations"
    ]
]

display(followup_sensors)

assert len(followup_sensors) == 10

,node_id,group,persistence_mae,observations
0,140,Highest Error,5.135753,23974
1,62,Highest Error,4.800096,23974
2,30,Highest Error,4.796995,23974
3,126,Highest Error,4.732942,23974
4,51,Highest Error,4.681790,23974
5,16,Lowest Error,1.808754,23974
6,74,Lowest Error,1.832561,23974
7,107,Lowest Error,1.898510,23974
8,123,Lowest Error,1.906989,23974
9,183,Lowest Error,1.974499,23974


In [16]:
tests = {
    "ID excluded from speed statistics":
        "node_id" not in speed_summary.index,

    "Only d0 speed columns summarized":
        all(
            col.endswith("_d0")
            for col in speed_summary.index
        ),

    "Zero-min/Q1 pattern tested":
        "stuck_zero_pattern"
        in speed_summary.columns,

    "207 sensors analyzed":
        len(sensor_mae) == 207,

    "Persistence errors are valid":
        (
            sensor_mae["persistence_mae"] >= 0
        ).all(),

    "Coefficient of variation valid":
        np.isfinite(mae_cv)
        and mae_cv >= 0,

    "5 highest-error sensors found":
        len(highest_5) == 5,

    "5 lowest-error sensors found":
        len(lowest_5) == 5
}


print("=" * 60)
print("STATISTICAL ANALYSIS VALIDATION")
print("=" * 60)

for name, passed in tests.items():

    print(
        f"{'PASS' if passed else 'FAIL':4} : {name}"
    )

print("=" * 60)


if all(tests.values()):

    print("\nALL TEST CASES PASSED")

    print(f"""
Speed columns analyzed       : {len(speed_cols)}
Sensors analyzed             : {len(sensor_mae)}
Stuck-zero-pattern columns   : {len(flagged_speed_cols)}
Mean persistence MAE         : {mae_mean:.4f} mph
Std of sensor MAE            : {mae_std:.4f} mph
Coefficient of variation     : {mae_cv:.4f}
Highest-error sensor         : {highest_5.iloc[0]['node_id']}
Lowest-error sensor          : {lowest_5.iloc[0]['node_id']}

Statistical and persistence-baseline analysis completed.
""")

else:

    failed = [
        name
        for name, passed in tests.items()
        if not passed
    ]

    print("\nVALIDATION FAILED")

    for name in failed:
        print(" -", name)

    raise AssertionError(
        f"{len(failed)} validation test(s) failed."
    )

STATISTICAL ANALYSIS VALIDATION
PASS : ID excluded from speed statistics
PASS : Only d0 speed columns summarized
PASS : Zero-min/Q1 pattern tested
PASS : 207 sensors analyzed
PASS : Persistence errors are valid
PASS : Coefficient of variation valid
PASS : 5 highest-error sensors found
PASS : 5 lowest-error sensors found

ALL TEST CASES PASSED

Speed columns analyzed       : 24
Sensors analyzed             : 207
Stuck-zero-pattern columns   : 24
Mean persistence MAE         : 2.9174 mph
Std of sensor MAE            : 0.6387 mph
Coefficient of variation     : 0.2189
Highest-error sensor         : 140.0
Lowest-error sensor          : 16.0

Statistical and persistence-baseline analysis completed.

